# Batch 8: recheck
Three checks from the audit of Batches 1-7.

**A. Hover on the shared split.** Batch 3 trained on doublet_1-5 and tested on doublet_6 only; the shared split (split.txt) trains on doublet_1-4 and tests on doublet_5 and doublet_6. Rerun Batch 3 (same design) on the shared split, physics constants from a simulation fit on doublet_1-4 only. Also: error vs horizon for every model.

**B. Hover damping xi = 0.** The hover simulation fit on Amr's clean windows gave xi = 0. Compare the two window choices.

**C. Closed-loop position: is the acceleration score robust?** Acceleration is a smoothed derivative of speed; the smoothing window changes the score. Rescore with 3 windows, keep every seed (median as well as mean).

## A. Hover on the shared split (Batch 3 design)

In [1]:
import numpy as np, pandas as pd, torch, time, matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import least_squares
from scipy.signal import savgol_filter
torch.set_num_threads(4)
D = '../data/1dof/'
FL = pd.read_csv(D + 'flights_1dof.csv').set_index('file')
TAU = 0.02; DT = 0.02
TRAIN = [f'doublet_{i}.csv' for i in range(1, 5)]
TESTS = ['doublet_5.csv', 'doublet_6.csv']
SEEDS = [0, 1, 2, 3, 4]; EPOCHS = 20000

In [2]:
def load(name):
    r = FL.loc[name]; df = pd.read_csv(D + name)
    after = df[df.t > r.inj_start_s + 0.7]
    end = after.t[after.M.abs() > 0.12].iloc[0] - 0.05            # just before the motor kick-back
    f = df[(df.t >= r.inj_start_s - 0.1 - TAU) & (df.t < end)].reset_index(drop=True)
    f['t'] = f.t - f.t[0] - TAU
    return f
def steps(f):
    t = np.arange(0, f.t.iloc[-1] - DT, DT)
    th = np.interp(t, f.t, f.theta); q = np.interp(t, f.t, f.q)
    M = np.array([np.mean(np.interp(np.linspace(ti, ti + DT, 6)[:-1] - TAU, f.t, f.M)) for ti in t])
    return t, th, q, M
flights = {n: load(n) for n in TRAIN + TESTS}
grid = {n: steps(f) for n, f in flights.items()}
print({n: round(f.t.iloc[-1], 2) for n, f in flights.items()})

{'doublet_1.csv': np.float64(1.46), 'doublet_2.csv': np.float64(1.46), 'doublet_3.csv': np.float64(1.45), 'doublet_4.csv': np.float64(1.45), 'doublet_5.csv': np.float64(1.46), 'doublet_6.csv': np.float64(1.46)}


In [3]:
def accel(theta, q, M, p):
    return -2 * p['xi'] * p['wn'] * q - p['wn']**2 * (theta - p['th0']) + p['mu'] * M

def euler_like_ivp(p):
    # equation model as a step function: exact ODE solve over one DT with M held constant
    def step(th, q, M):
        y = solve_ivp(lambda t, z: [z[1], accel(z[0], z[1], M, p)], (0, DT), [th, q], max_step=DT / 5).y[:, -1]
        return y[0], y[1]
    return step

def vaf(y, yhat):
    return max(1 - np.var(y - yhat) / np.var(y), 0) * 100

def run_ivp(p, f):
    t, M = f.t.values, f.M.values
    rhs = lambda tt, z: [z[1], accel(z[0], z[1], np.interp(tt - TAU, t, M), p)]
    i0 = np.searchsorted(t, 0.0)
    return solve_ivp(rhs, (t[i0], t[-1]), [f.theta[i0], f.q[i0]], t_eval=t[i0:], max_step=0.004).y[0], i0
keys = ['wn', 'xi', 'mu', 'th0']
def residual(v):
    out = []
    for n in TRAIN:
        y, i0 = run_ivp(dict(zip(keys, v)), flights[n]); out.append(y - flights[n].theta.values[i0:])
    return np.concatenate(out)
fit = least_squares(residual, [3.8, 0.15, 180, -0.15], bounds=([0.5, 0, 0, -0.5], [20, 2, 2000, 0.5]))
P_SIM = dict(zip(keys, fit.x)); P_PINN = P_SIM
print('hover sim fit on doublet_1-4:', {k: round(v, 4) for k, v in P_SIM.items()})

hover sim fit on doublet_1-4: {'wn': np.float64(3.748), 'xi': np.float64(0.1158), 'mu': np.float64(190.1398), 'th0': np.float64(-0.1705)}


In [4]:
def mlp(n_in, n_out, width=32, depth=3):
    layers, n = [], n_in
    for _ in range(depth):
        layers += [torch.nn.Linear(n, width), torch.nn.Tanh()]; n = width
    return torch.nn.Sequential(*layers, torch.nn.Linear(n, n_out))

col = lambda x: torch.tensor(np.asarray(x), dtype=torch.float32).reshape(-1, 1)
S = dict(th=0.3, q=4.0, M=0.06)          # normalisation scales (inputs and outputs about -1..1)

def pinc_model(net):
    def step(th, q, M, dt):              # all torch columns; returns (theta, q) after dt
        x = torch.cat([th / S['th'], q / S['q'], M / S['M'], dt / DT], 1)
        o = net(x)
        return S['th'] * o[:, :1], S['q'] * o[:, 1:2]
    return step

In [5]:
def train_pinc(names, seed=0, epochs=EPOCHS, lr=1e-3, w_phys=1.0, n_col=2000, p=P_PINN):
    # data
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    G = [grid[n] for n in names]
    th0 = col(np.concatenate([g[1][:-1] for g in G])); q0 = col(np.concatenate([g[2][:-1] for g in G]))
    th1 = col(np.concatenate([g[1][1:] for g in G])); q1 = col(np.concatenate([g[2][1:] for g in G]))
    M0 = col(np.concatenate([g[3][:-1] for g in G]))
    # network
    net = mlp(4, 2); step = pinc_model(net)
    # optimizer
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=lr / 20)
    # loop
    hist = []
    for e in range(epochs):
        a, b = step(th0, q0, M0, torch.full_like(th0, DT))                    # one step of DT
        data = (((a - th1) / S['th'])**2).mean() + (((b - q1) / S['q'])**2).mean()
        a, b = step(th0, q0, M0, torch.zeros_like(th0))                       # dt = 0 -> same state
        start = (((a - th0) / S['th'])**2).mean() + (((b - q0) / S['q'])**2).mean()
        physics = torch.tensor(0.0)
        if w_phys > 0:
            zt = col(rng.uniform(-0.35, 0.35, n_col)); zq = col(rng.uniform(-4, 4, n_col))
            zM = col(rng.uniform(-0.06, 0.06, n_col)); dt = col(rng.uniform(0, DT, n_col)).requires_grad_(True)
            a, b = step(zt, zq, zM, dt)
            da = torch.autograd.grad(a, dt, torch.ones_like(a), create_graph=True)[0]    # d theta / d dt
            db = torch.autograd.grad(b, dt, torch.ones_like(b), create_graph=True)[0]    # d q / d dt
            physics = (((da - b) / S['q'])**2).mean() + (((db - accel(a, b, zM, p)) / (S['q'] / DT))**2).mean()
            a, b = step(zt, zq, zM, torch.zeros_like(zt))
            start = start + (((a - zt) / S['th'])**2).mean() + (((b - zq) / S['q'])**2).mean()
        loss = data + start + w_phys * physics
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
        if e % 1000 == 0 or e == epochs - 1:
            hist.append((e, data.item(), start.item(), physics.item()))
    return dict(net=net, step=step, history=pd.DataFrame(hist, columns=['epoch', 'data', 'start', 'physics']))

In [6]:
def run_open(step_fn, th_start, q_start, M_list, is_torch=True):
    # chain the model: each output is the next input
    th, q = th_start, q_start; out = [th]
    for M in M_list[:-1]:
        if is_torch:
            with torch.no_grad():
                a, b = step_fn(col([th]), col([q]), col([M]), col([DT])); th, q = a.item(), b.item()
        else:
            th, q = step_fn(th, q, M)
        out.append(th)
    return np.array(out)

def run_one_step(step_fn, g, is_torch=True):
    t, th, q, M = g; out = [th[0]]
    for k in range(len(t) - 1):
        if is_torch:
            with torch.no_grad(): a, _ = step_fn(col([th[k]]), col([q[k]]), col([M[k]]), col([DT])); out.append(a.item())
        else:
            out.append(step_fn(th[k], q[k], M[k])[0])
    return np.array(out)

In [7]:
runs = {'PINC': [], 'plain NN': []}
for s in SEEDS:
    for label, wp in [('PINC', 1.0), ('plain NN', 0.0)]:
        t0 = time.time(); r = train_pinc(TRAIN, seed=s, w_phys=wp); runs[label].append(r)
        print(label, 'seed', s, f'{time.time() - t0:.0f} s')
HOR = [0.25, 0.5, 1.0]
rows = []
for n in TESTS:
    g = grid[n]; t, th, q, M = g
    models = {'Equation, sim-fit constants': [run_open(euler_like_ivp(P_SIM), th[0], q[0], M, False)]}
    for label, rs in runs.items(): models[label] = [run_open(r['step'], th[0], q[0], M) for r in rs]
    for label, ys in models.items():
        v = [vaf(th, y) for y in ys]
        e = {f'RMSE first {h} s [deg]': np.mean([np.degrees(np.sqrt(np.mean((y[:int(h / DT) + 1] - th[:int(h / DT) + 1])**2))) for y in ys]) for h in HOR}
        rows.append(dict(test=n[:-4], model=label, VAF_mean=np.mean(v), VAF_std=np.std(v), VAF_min=np.min(v), **e))
A = pd.DataFrame(rows).round(2); A.to_csv('../results/batch8_hover_shared_split.csv', index=False); A

PINC seed 0 107 s


plain NN seed 0 30 s


PINC seed 1 106 s


plain NN seed 1 31 s


PINC seed 2 102 s


plain NN seed 2 30 s


PINC seed 3 103 s


plain NN seed 3 31 s


PINC seed 4 106 s


plain NN seed 4 32 s


,test,model,VAF_mean,VAF_std,VAF_min,RMSE first 0.25 s [deg],RMSE first 0.5 s [deg],RMSE first 1.0 s [deg]
0,doublet_5,"Equation, sim-fit constants",56.07,0.00,56.07,0.12,1.99,2.65
1,doublet_5,PINC,65.56,1.70,62.56,0.15,1.77,2.49
2,doublet_5,plain NN,58.93,2.23,55.26,0.35,1.98,2.70
3,doublet_6,"Equation, sim-fit constants",97.40,0.00,97.40,0.15,1.31,2.00
4,doublet_6,PINC,98.90,0.17,98.59,0.18,1.03,1.81
5,doublet_6,plain NN,97.88,0.30,97.35,0.39,1.17,1.61


In [8]:
g = grid['doublet_6.csv']; t, th, q, M = g
new = {'doublet x1.5': M[0] + 1.5 * (M - M[0]), 'hold 0.03 for 1 s': np.where((t > 0.1) & (t < 1.1), M[0] + 0.03, M[0])}
rows2 = []
for name, Mn in new.items():
    ref = run_open(euler_like_ivp(P_SIM), th[0], q[0], Mn, False)
    for label in ['PINC', 'plain NN']:
        v = [vaf(ref, run_open(r['step'], th[0], q[0], Mn)) for r in runs[label]]
        rows2.append(dict(input=name, model=label, VAF_mean=np.mean(v), VAF_std=np.std(v), VAF_min=np.min(v)))
A2 = pd.DataFrame(rows2).round(2); A2.to_csv('../results/batch8_hover_new_inputs.csv', index=False); A2

,input,model,VAF_mean,VAF_std,VAF_min
0,doublet x1.5,PINC,99.71,0.09,99.55
1,doublet x1.5,plain NN,97.42,0.55,96.62
2,hold 0.03 for 1 s,PINC,95.33,1.43,92.83
3,hold 0.03 for 1 s,plain NN,76.85,5.35,71.54


## B. Hover damping: my windows vs the clean-file windows

In [9]:
C1 = '../data/clean/1dof/'
def fit_on(frames):
    def res(v):
        p = dict(zip(keys, v)); out = []
        for f in frames:
            tt, Mv = f.t.values, f.M.values
            rhs = lambda x, z: [z[1], accel(z[0], z[1], np.interp(x - TAU, tt, Mv), p)]
            out.append(solve_ivp(rhs, (tt[0], tt[-1]), [f.theta[0], f.q[0]], t_eval=tt, max_step=0.004).y[0] - f.theta.values)
        return np.concatenate(out)
    return dict(zip(keys, least_squares(res, [3.8, 0.15, 180, 0], bounds=([0.5, 0, 0, -0.5], [20, 2, 2000, 0.5])).x))
clean = [pd.read_csv(C1 + f'doublet_{i}.csv') for i in range(1, 5)]
B = pd.DataFrame({'my windows (until motor kick-back)': P_SIM, 'clean windows (fixed 1.5 s)': fit_on(clean),
                  'clean windows cut at 1.2 s': fit_on([f[f.t < 1.2] for f in clean])}).T.round(4)
B['window length [s]'] = [np.mean([flights[n].t.iloc[-1] for n in TRAIN]).round(2), 1.5, 1.2]
B.to_csv('../results/batch8_hover_damping_windows.csv'); B

,wn,xi,mu,th0,window length [s]
my windows (until motor kick-back),3.7480,0.1158,190.1398,-0.1705,1.46
clean windows (fixed 1.5 s),4.3873,0.0000,122.0239,-0.0298,1.50
clean windows cut at 1.2 s,5.4595,0.0000,119.5924,0.0134,1.20


## C. Closed-loop position: acceleration score vs smoothing, every seed

In [10]:
torch.set_num_threads(4)
D2 = '../data/clean/2dof/'
EPOCHS7 = 10000
grid = lambda n: pd.read_csv(D2 + n + '.csv').iloc[::5].reset_index(drop=True)      # 50 Hz
def att_window(f):
    k = np.flatnonzero(np.abs(f.theta_sp.values) > 0.05)[0]; t0 = f.t[k]
    return f[(f.t >= t0 - 1) & (f.t < t0 + 3)].reset_index(drop=True)
TASKS = {
 'CL attitude': dict(y='theta', yd='q', r='theta_sp', train=['attitude_doublet_1', 'attitude_doublet_3'],
                     test=['attitude_doublet_4', 'attitude_doublet_5'], cut=att_window, thesis=(16.16, 0.56, 0.0),
                     thesis_vaf='pitch 99.86, rate 97.27'),
 'CL position': dict(y='x', yd='v', r='x_sp', train=['position_1', 'position_3'], test=['position_2'],
                     cut=lambda f: f, thesis=(3.47, 0.84, 0.0), thesis_vaf='x 99.45, acc 58.72')}
F = {n: cfg['cut'](grid(n)) for cfg in TASKS.values() for n in cfg['train'] + cfg['test']}
vaf = lambda y, yh: max(1 - np.var(y - yh) / np.var(y), 0) * 100
acc = lambda v: savgol_filter(v, 25, 2, deriv=1, delta=DT)          # smoothed derivative

In [11]:
def accel7(y, yd, r, p):
    wn, xi = p[0], p[1]
    return wn**2 * (r - y) - 2 * xi * wn * yd

def run_ivp(p, f, c):
    t, R = f.t.values, f[c['r']].values
    rhs = lambda tt, z: [z[1], accel7(z[0], z[1], np.interp(tt - p[2], t, R), p)]
    return solve_ivp(rhs, (t[0], t[-1]), [f[c['y']][0], f[c['yd']][0]], t_eval=t, max_step=0.005, rtol=1e-8, atol=1e-10).y

def grey_fit(c):
    res = lambda p: np.concatenate([run_ivp(p, F[n], c)[0] - F[n][c['y']].values for n in c['train']])
    r = least_squares(res, list(c['thesis'][:2]) + [0.02], bounds=([0.5, 0, 0], [60, 3, 0.3]), diff_step=1e-3)
    J = r.jac; s2 = (r.fun**2).sum() / (len(r.fun) - 3)
    return r.x, np.sqrt(np.diag(s2 * np.linalg.inv(J.T @ J)))

In [12]:
def mlp7(n_in, n_out, width=32, depth=3):
    layers, n = [], n_in
    for _ in range(depth):
        layers += [torch.nn.Linear(n, width), torch.nn.Tanh()]; n = width
    return torch.nn.Sequential(*layers, torch.nn.Linear(n, n_out))
col = lambda x: torch.tensor(np.asarray(x), dtype=torch.float32).reshape(-1, 1)

def pairs7(names, c):
    out = []
    for n in names:
        f = F[n]; y, yd, r = f[c['y']].values, f[c['yd']].values, f[c['r']].values
        out.append(np.column_stack([y[1:-1], yd[1:-1], r[1:-1], r[:-2], y[2:], yd[2:]]))
    return np.vstack(out)

def pinc_model7(net, sc):
    def step(y, yd, r, rp, dt):
        x = torch.cat([(y - sc['m'][0]) / sc['s'][0], (yd - sc['m'][1]) / sc['s'][1],
                       (r - sc['m'][2]) / sc['s'][2], (rp - sc['m'][3]) / sc['s'][3], dt / DT], 1)
        o = net(x)
        return y + sc['d'][0] * o[:, :1], yd + sc['d'][1] * o[:, 1:2]
    return step

def train_pinc7(c, p, seed=0, epochs=EPOCHS7, lr=1e-3, w_phys=1.0, n_col=2000):
    # data
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    A = pairs7(c['train'], c)
    sc = dict(m=A[:, :4].mean(0), s=A[:, :4].std(0) + 1e-6, d=(A[:, 4:6] - A[:, :2]).std(0))
    y0, v0, r0, rp0, y1, v1 = [col(A[:, i]) for i in range(6)]
    lo, hi = A[:, :4].min(0), A[:, :4].max(0); pad = 0.2 * (hi - lo)
    # network
    net = mlp7(5, 2); step = pinc_model7(net, sc)
    # optimizer
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs, eta_min=lr / 20)
    # loop
    for e in range(epochs):
        a, b = step(y0, v0, r0, rp0, torch.full_like(y0, DT))
        data = (((a - y1) / sc['d'][0])**2).mean() + (((b - v1) / sc['d'][1])**2).mean()
        a, b = step(y0, v0, r0, rp0, torch.zeros_like(y0))
        start = (((a - y0) / sc['d'][0])**2).mean() + (((b - v0) / sc['d'][1])**2).mean()
        physics = torch.tensor(0.0)
        if w_phys > 0:
            z = [col(rng.uniform(lo[i] - pad[i], hi[i] + pad[i], n_col)) for i in range(4)]
            dt = col(rng.uniform(0, DT, n_col)).requires_grad_(True)
            a, b = step(*z, dt)
            da = torch.autograd.grad(a, dt, torch.ones_like(a), create_graph=True)[0]
            db = torch.autograd.grad(b, dt, torch.ones_like(b), create_graph=True)[0]
            w = min(p[2] / DT, 1.0)                                         # delayed setpoint inside the step
            rt = z[2] - w * (1 - dt / DT) * (z[2] - z[3])
            physics = (((da - b) * DT / sc['d'][0])**2).mean() + (((db - accel7(a, b, rt, p)) * DT / sc['d'][1])**2).mean()
        loss = data + start + w_phys * physics
        opt.zero_grad(); loss.backward(); opt.step(); sched.step()
    return dict(step=step, last=dict(data=data.item(), start=start.item(), physics=physics.item()))

def run_open7(step, f, c):
    y, yd, r = f[c['y']].values, f[c['yd']].values, f[c['r']].values
    a, b = y[1], yd[1]; Y, V = [y[0], a], [yd[0], b]
    with torch.no_grad():
        for k in range(1, len(f) - 1):
            a, b = step(col([a]), col([b]), col([r[k]]), col([r[k - 1]]), col([DT])); a, b = a.item(), b.item()
            Y.append(a); V.append(b)
    return np.array(Y), np.array(V)

In [13]:
c = TASKS['CL position']; p, sd = grey_fit(c); f = F['position_2']; v = f.v.values
WIN = [11, 25, 51]                                     # smoothing windows: 0.22, 0.5, 1.02 s
acc_w = lambda s, w: savgol_filter(s, w, 2, deriv=1, delta=DT)
rows3 = []
def add(label, seed, V, X):
    rows3.append(dict(model=label, seed=seed, VAF_x=vaf(f.x.values, X), VAF_v=vaf(v, V), **{f'VAF_acc_{w * DT:.2f}s': vaf(acc_w(v, w), acc_w(V, w)) for w in WIN}))
Y = run_ivp(p, f, c); add('grey-box refit', '-', Y[1], Y[0])
Y = run_ivp(c['thesis'], f, c); add('Ghezzi constants', '-', Y[1], Y[0])
for label, wp in [('PINC', 1.0), ('plain NN', 0.0)]:
    for s in SEEDS:
        r = train_pinc7(c, p, seed=s, w_phys=wp); X, V = run_open7(r['step'], f, c); add(label, s, V, X)
Cc = pd.DataFrame(rows3).round(2); Cc.to_csv('../results/batch8_position_acc_smoothing.csv', index=False)
display(Cc)
Cc.drop(columns='seed').groupby('model').agg(['median', 'mean', 'min']).round(2)

,model,seed,VAF_x,VAF_v,VAF_acc_0.22s,VAF_acc_0.50s,VAF_acc_1.02s
0,grey-box refit,-,99.28,91.80,51.05,68.54,81.42
1,Ghezzi constants,-,92.88,27.55,0.00,0.00,0.00
2,PINC,0,99.40,95.77,79.95,86.34,91.29
3,PINC,1,99.35,96.15,80.71,87.26,92.04
4,PINC,2,99.37,96.03,80.90,86.87,91.41
5,PINC,3,99.31,96.01,81.30,87.38,91.67
6,PINC,4,99.33,95.80,79.34,85.88,91.06
7,plain NN,0,99.61,98.26,95.38,96.51,97.38
8,plain NN,1,99.64,98.20,94.92,96.10,97.07
9,plain NN,2,99.68,98.36,95.16,96.40,97.32


VAF_x                VAF_v               VAF_acc_0.22s  \
                 median   mean    min median   mean    min        median   
model                                                                      
Ghezzi constants  92.88  92.88  92.88  27.55  27.55  27.55          0.00   
PINC              99.35  99.35  99.31  96.01  95.95  95.77         80.71   
grey-box refit    99.28  99.28  99.28  91.80  91.80  91.80         51.05   
plain NN          99.61  79.71   0.00  98.26  78.62   0.00         95.16   

                               VAF_acc_0.50s               VAF_acc_1.02s  \
                   mean    min        median   mean    min        median   
model                                                                      
Ghezzi constants   0.00   0.00          0.00   0.00   0.00          0.00   
PINC              80.44  79.34         86.87  86.75  85.88         91.41   
grey-box refit    51.05  51.05         68.54  68.54  68.54         81.42   
plain NN          76.13   0.00         96.39  77.08   0.00         97.31   

                                
                   mean    min  
model                           
Ghezzi constants   0.00   0.00  
PINC              91.49  91.06  
grey-box refit    81.42  81.42  
plain NN          77.82   0.00